## Binder quick start (safe defaults)

- Optional: set `INAT_TOKEN` in this session for higher limits.
- Start with low-traffic settings: `per_page=25`, `max_pages=2`, `fetch_images=False`.
- This notebook runs without a token (anonymous), but may be rate-limited.

In [ ]:
import os
#from pynat.helpers import coming_soon, get_park_data
from helpers import coming_soon, get_park_data

INAT_TOKEN_SET = bool(os.getenv("INAT_TOKEN"))
print(f"INAT token set: {INAT_TOKEN_SET}")
print("Low-traffic defaults: per_page=25, max_pages=2, fetch_images=False")

# Example usage (uncomment to run)
# df = coming_soon(
#     "birds",
#     loc=(37.6669, -77.8883, 25),
#     per_page=25,
#     max_pages=2,
#     fetch_images=False,
# )
# display(df.head(20))

In [ ]:
import pandas as pd
import pyinaturalist as inat

In [ ]:
# get geogenters for all parks

parks = {
    'tucker': (37.66713, -77.88739, 0.4), 
    'hiddenrock': (37.70219, -77.87333, 0.63), 
    'leakesmill': (37.70678, -77.95971, 0.68), 
    'matthews': (37.78714, -78.03979, 0.34),
}

In [ ]:
res = {}
for park in parks:
    pagenum = 0
    res[park] = pd.DataFrame()
    while(True):
        pagenum += 1
        tmp = pd.json_normalize(inat.get_observation_species_counts(lat=parks[park][0], 
                                                                    lng=parks[park][1], 
                                                                    radius=parks[park][2], 
                                                                    verifiable=True,
                                                                    #capitve
                                                                    #introduced
                                                                    #native
                                                                    #endemic
                                                                    #threatened
                                                                    page=pagenum,
                                                                   )['results'])
        if tmp.empty:
            break
        res[park] = pd.concat([res[park], tmp])    
    normer = pd.json_normalize(inat.get_observation_species_counts(taxon_id=res[park]['taxon.id'].to_list(),
                                                                   place_id=1297, # relative to Virginia
                                                                   verifiable=True,
                                                                  )['results'])
    res[park]['normalizer'] = res[park]['taxon.id'].map(normer.set_index('taxon.id')['count'])
    res[park]['sorter'] = res[park]['count']/res[park]['normalizer']
    res[park].sort_values('sorter', ascending=False, inplace=True)

### suspicious that so many of these max at 500...

In [ ]:
for park in parks:
    print(f"{park}: {res[park].shape}")

In [ ]:
res['tucker'][['count', 'taxon.name', 'taxon.preferred_common_name', 'taxon.wikipedia_url']].sort_values('count', ascending=False).head(25)

In [ ]:
res['leakesmill'][['count', 'taxon.name', 'taxon.preferred_common_name', 'taxon.wikipedia_url']].sort_values('count', ascending=False).head(25)

In [ ]:
res['hiddenrock'][['count', 'taxon.name', 'taxon.preferred_common_name', 'taxon.wikipedia_url']].sort_values('count', ascending=False).head(25)

In [ ]:
res['matthews'][['count', 'taxon.name', 'taxon.preferred_common_name', 'taxon.wikipedia_url']].sort_values('count', ascending=False).head(25)

In [ ]:
# highlight unusual species esp. engagered ones (sum and check trends)

In [ ]:
# highlight unusual or *unusually-increasing* invasives (sum and check trends)